# Automated Medical Image Analysis for Early Disease Detection
## Cloud GPU Training Notebook: RSNA Pneumonia Detection using YOLOv8

This notebook is designed for **Kaggle Notebooks (GPU T4/P100)** or **Google Colab (T4 GPU)**.

### Pipeline Overview:
1. **Environment Setup & GPU Check**: Verify CUDA availability and install medical imaging dependencies.
2. **Dataset Ingestion**: Read RSNA DICOM chest radiographs (`.dcm`) and bounding box annotations (`stage_2_train_labels.csv`).
3. **Preprocessing & Conversion**: Apply DICOM windowing/VOI LUT, convert pixel arrays to images, and convert annotations to normalized YOLO format `[class, x_center, y_center, width, height]`.
4. **Dataset Configuration**: Generate `rsna_yolo.yaml` specifying train/val splits and classes.
5. **YOLOv8 Training**: Fine-tune YOLOv8 on chest radiographs.
6. **Evaluation & Visualization**: Compute mAP50, mAP50-95, draw predicted vs ground-truth bounding boxes.
7. **Artifact Export**: Export `best.pt` for local inference on Apple M1 Mac.

### 1. Environment Setup & GPU Check

In [ ]:
!nvidia-smi
!pip install -q ultralytics pydicom opencv-python-headless scikit-learn matplotlib pandas numpy

In [ ]:
import os
import torch
import pydicom
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")

### 2. Locate RSNA Dataset Files
On Kaggle, add the `rsna-pneumonia-detection-challenge` dataset to the notebook.
The data path is typically `/kaggle/input/rsna-pneumonia-detection-challenge`.

In [ ]:
KAGGLE_PATH = Path("/kaggle/input/rsna-pneumonia-detection-challenge")
COLAB_PATH = Path("./data/raw")

DATA_DIR = KAGGLE_PATH if KAGGLE_PATH.exists() else COLAB_PATH
print(f"Using dataset directory: {DATA_DIR}")

labels_csv = DATA_DIR / "stage_2_train_labels.csv"
if labels_csv.exists():
    df = pd.read_csv(labels_csv)
    print(f"Loaded labels. Total rows: {len(df)}")
    display(df.head())
else:
    print(f"Labels file not found at {labels_csv}. Ensure dataset is attached/downloaded.")

### 3. Preprocessing: DICOM to YOLO Format
YOLO expects:
- Images saved as `.jpg` or `.png`.
- Corresponding `.txt` files with normalized coordinates: `<class_id> <x_center> <y_center> <width> <height>` (values between 0.0 and 1.0).
- Class `0`: Lung Opacity (Pneumonia).

In [ ]:
# Preprocessing and conversion routine
OUTPUT_DIR = Path("/kaggle/working/rsna_yolo") if Path("/kaggle/working").exists() else Path("./data/processed/rsna_yolo")
(OUTPUT_DIR / "images" / "train").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "images" / "val").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "labels" / "train").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "labels" / "val").mkdir(parents=True, exist_ok=True)

print(f"Output directory configured at: {OUTPUT_DIR}")

### 4. Create YOLO Dataset Configuration (`dataset.yaml`)

In [ ]:
yaml_content = f"""path: {OUTPUT_DIR.resolve()}
train: images/train
val: images/val

nc: 1
names: ['Lung Opacity']
"""

yaml_path = OUTPUT_DIR / "dataset.yaml"
with open(yaml_path, "w") as f:
    f.write(yaml_content)

print(f"Dataset YAML written to {yaml_path}:")
print(yaml_content)

### 5. Train YOLOv8 Model
We use pretrained weights `yolov8s.pt` (small) or `yolov8n.pt` (nano) and fine-tune for object detection.

In [ ]:
# Initialize model
model = YOLO("yolov8s.pt")

# Start training
# results = model.train(
#     data=str(yaml_path),
#     epochs=25,
#     imgsz=640,
#     batch=16,
#     device=0,
#     project="rsna_pneumonia",
#     name="yolov8s_run",
#     save=True
# )
print("Training configuration ready. Uncomment to execute training on Cloud GPU.")

### 6. Evaluate Model on Validation Set

In [ ]:
# Validate metrics (mAP50, mAP50-95)
# metrics = model.val()
# print(f"mAP@50: {metrics.box.map50:.4f}")
# print(f"mAP@50-95: {metrics.box.map:.4f}")

### 7. Download Weights for Apple M1 Deployment
Once training completes, the best model checkpoint is saved to:
`runs/detect/rsna_pneumonia/yolov8s_run/weights/best.pt`.

Download this file and copy it to your local project directory under `models/weights/best.pt`.